In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output

JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module — file is CRUD_Python_Module.py in the same folder
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "Aacuser123!"
shelter = AnimalShelter(username, password)

# Retrieve all documents from MongoDB into a Pandas DataFrame
df = pd.DataFrame.from_records(shelter.read({}))

# Drop the '_id' column added by MongoDB to prevent DataTable crash
df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################

app = JupyterDash('GraziosoSalvareDashboard')

app.layout = html.Div([

    # Unique identifier header
    html.Center([
        html.B(html.H1('Grazioso Salvare Animal Rescue Dashboard')),
        html.P('Developed by Juan — SNHU CS-340'),
    ]),
    html.Hr(),

    # Interactive DataTable
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),

        # Required for map callback to function
        row_selectable="single",
        selected_rows=[0],

        # User-friendly features
        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",

        # Styling
        style_table={'overflowX': 'auto'},
        style_header={
            'backgroundColor': 'rgb(30, 30, 30)',
            'color': 'white',
            'fontWeight': 'bold'
        },
        style_cell={
            'backgroundColor': 'rgb(50, 50, 50)',
            'color': 'white',
            'textAlign': 'left',
            'minWidth': '100px'
        },
    ),

    html.Br(),
    html.Hr(),

    # Geolocation chart placeholder
    html.Div(
        id='map-id',
        className='col s12 m6',
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback 1: Highlight selected column in the DataTable
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]


# Callback 2: Update geolocation chart based on selected DataTable row
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    # Guard against no data on initial load
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Default to first row if no row selected
    row = 0 if index is None else index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],  # Austin, TX
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                # Column 13 = location_lat, Column 14 = location_long
                # Column 4  = breed, Column 9 = animal name
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app inline in JupyterLab
# If port 8050 is taken, change to 8051 or 8052
app.run_server(mode='inline', port=8050)